<a href="https://colab.research.google.com/github/AlvaroAla/TE-IA/blob/main/aula1_0_mini_transformer_optimus_prime.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GSI073 - Tópicos Especiais de Inteligência Artificial (Large Language Models) - Prof. Marcelo Keese Albertini

Este código foi escrito em aula para demonstrar rapidamente como é a arquitetura de um Transformer.

In [ ]:
import torch
from torch import nn

from torch.nn import functional as F

In [ ]:
class MeuBloco(nn.Module):
  def __init__(self, n_heads, model_dim, vocab_size):
    super().__init__()
    self.norm1 = nn.LayerNorm(model_dim, bias = False)
    self.norm2 = nn.LayerNorm(model_dim, bias = False)

    self.attention = nn.MultiheadAttention(embed_dim = model_dim,
                                           num_heads = n_heads
                                           )
    self.ffn = nn.Sequential(nn.Linear(model_dim, 2*model_dim),
                             nn.ReLU(),
                             nn.Linear(2*model_dim, model_dim))

  def forward(self, x):

    res_atencao, _ = self.attention(x, x, x)
    x = self.norm1(x + res_atencao)

    res_ffn = self.ffn(x)
    x = self.norm2(res_ffn + x)
    return x



class MeuEncoder(nn.Module):
  def __init__(self, n_layers, n_heads, model_dim, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, model_dim)
    self.layers = nn.ModuleList( [
        MeuBloco(n_heads, model_dim, vocab_size)
        for _ in range(n_heads) ])

  def forward(self, x):
    x = self.embedding(x)

    for layer in self.layers:
      x = layer(x)

    return x



In [ ]:
mini_llm = MeuEncoder(n_layers = 2, n_heads = 2, model_dim = 8, vocab_size= 10)

In [ ]:
mini_llm(torch.tensor([0,1,3,4]))

In [ ]:
mini_llm

In [ ]:
mini_llm.embedding.weight.requires_grad = False

In [ ]:
mini_llm.embedding.weight

In [ ]:
class MeuBlocoDecoder(nn.Module):
  def __init__(self, model_dim, vocab_size, n_heads, n_layers):
    super().__init__()
    self.norm1 = nn.LayerNorm(model_dim)
    self.norm2 = nn.LayerNorm(model_dim)
    self.norm3 = nn.LayerNorm(model_dim)

    self.embedding = nn.Embedding(vocab_size, model_dim)

    self.att_cros = nn.MultiheadAttention(model_dim, n_heads)
    self.att_self = nn.MultiheadAttention(model_dim, n_heads)

    self.ffn = nn.Sequential(nn.Linear(model_dim, 2*model_dim), nn.ReLU(), nn.Linear(2*model_dim, model_dim))

    self.lm_head = nn.Linear(model_dim, vocab_size)

    self.softmax = nn.Softmax(0)

  def forward(self, x, encoder_output):
    x = self.embedding(x)

    res_att_self = self.att_self(x, x, x)
    x = self.norm1(x + res_att_self)

    res_att_cros = self.att_cros(encoder_output, encoder_output, x)
    x = self.norm2( x + res_att_cros)

    res_ffn = self.ffn(x)

    x = self.norm3(x + res_ffn)

    logits = self.softmax(self.lm_head(x))

    return logits


    logits = nn.Softmax(out)



# Atividade – Arquitetura básica de um Transformer

Nesta atividade, o objetivo foi entender de forma prática como funciona a arquitetura básica de um **Transformer**, utilizando PyTorch.

O código apresentado em aula mostra uma versão simplificada de um encoder e de um decoder, com os principais componentes da arquitetura, como:

- **Embedding**, para transformar tokens em vetores;
- **Multi-Head Attention**, para capturar relações entre os elementos da sequência;
- **Layer Normalization**, para estabilizar o treinamento;
- **Feed Forward Network (FFN)**, para processar melhor as representações;
- **Camada final de saída**, para gerar probabilidades sobre o vocabulário.

No encoder, os tokens passam pela camada de embedding e depois por blocos com atenção e rede feed forward.  
No decoder, além da self-attention, também existe a cross-attention, que usa a saída do encoder para ajudar na geração da saída.

Com essa atividade, foi possível visualizar como os dados passam pelas camadas e como a arquitetura do Transformer é organizada de forma modular.

In [ ]:
import torch
from torch import nn

# ----------------------------
# Bloco do Encoder
# ----------------------------
class MeuBloco(nn.Module):
    def __init__(self, n_heads, model_dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(model_dim)
        self.norm2 = nn.LayerNorm(model_dim)

        self.attention = nn.MultiheadAttention(
            embed_dim=model_dim,
            num_heads=n_heads,
            batch_first=True
        )

        self.ffn = nn.Sequential(
            nn.Linear(model_dim, 2 * model_dim),
            nn.ReLU(),
            nn.Linear(2 * model_dim, model_dim)
        )

    def forward(self, x):
        res_atencao, _ = self.attention(x, x, x)
        x = self.norm1(x + res_atencao)

        res_ffn = self.ffn(x)
        x = self.norm2(x + res_ffn)

        return x


# ----------------------------
# Encoder
# ----------------------------
class MeuEncoder(nn.Module):
    def __init__(self, n_layers, n_heads, model_dim, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, model_dim)

        self.layers = nn.ModuleList([
            MeuBloco(n_heads, model_dim)
            for _ in range(n_layers)
        ])

    def forward(self, x):
        x = self.embedding(x)  # [batch, seq_len, model_dim]

        for layer in self.layers:
            x = layer(x)

        return x


# ----------------------------
# Bloco do Decoder
# ----------------------------
class MeuBlocoDecoder(nn.Module):
    def __init__(self, model_dim, vocab_size, n_heads):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, model_dim)

        self.norm1 = nn.LayerNorm(model_dim)
        self.norm2 = nn.LayerNorm(model_dim)
        self.norm3 = nn.LayerNorm(model_dim)

        self.att_self = nn.MultiheadAttention(
            embed_dim=model_dim,
            num_heads=n_heads,
            batch_first=True
        )

        self.att_cross = nn.MultiheadAttention(
            embed_dim=model_dim,
            num_heads=n_heads,
            batch_first=True
        )

        self.ffn = nn.Sequential(
            nn.Linear(model_dim, 2 * model_dim),
            nn.ReLU(),
            nn.Linear(2 * model_dim, model_dim)
        )

        self.lm_head = nn.Linear(model_dim, vocab_size)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, encoder_output):
        x = self.embedding(x)  # [batch, seq_len, model_dim]

        res_att_self, _ = self.att_self(x, x, x)
        x = self.norm1(x + res_att_self)

        res_att_cross, _ = self.att_cross(x, encoder_output, encoder_output)
        x = self.norm2(x + res_att_cross)

        res_ffn = self.ffn(x)
        x = self.norm3(x + res_ffn)

        logits = self.lm_head(x)
        probs = self.softmax(logits)

        return probs


# ----------------------------
# Parâmetros
# ----------------------------
vocab_size = 10
model_dim = 8
n_heads = 2
n_layers = 2

# Modelos
encoder = MeuEncoder(n_layers=n_layers, n_heads=n_heads, model_dim=model_dim, vocab_size=vocab_size)
decoder = MeuBlocoDecoder(model_dim=model_dim, vocab_size=vocab_size, n_heads=n_heads)

In [ ]:
# Exemplo de entrada
entrada_encoder = torch.tensor([[0, 1, 3, 4]])   # batch=1, seq_len=4
entrada_decoder = torch.tensor([[1, 2, 3, 4]])   # batch=1, seq_len=4

# Saída do encoder
saida_encoder = encoder(entrada_encoder)
print("Saída do encoder:")
print(saida_encoder)
print("Shape da saída do encoder:", saida_encoder.shape)

# Congelando embedding do encoder
encoder.embedding.weight.requires_grad = False
print("\nEmbedding congelado:", encoder.embedding.weight.requires_grad)

# Saída do decoder
saida_decoder = decoder(entrada_decoder, saida_encoder)
print("\nSaída do decoder:")
print(saida_decoder)
print("Shape da saída do decoder:", saida_decoder.shape)

# Mostrando os modelos
print("\nEstrutura do encoder:")
print(encoder)

print("\nEstrutura do decoder:")
print(decoder)

Foi possível perceber que o encoder gera representações vetoriais da sequência de entrada, enquanto o decoder utiliza essas representações para produzir a saída no espaço do vocabulário. Dessa forma, a atividade ajudou a entender melhor os blocos principais que formam um Transformer.